In [1]:
import os
import gc
import numpy as np
import torch
import torch.nn.functional as F
import itertools
from itertools import cycle
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline, T2IAdapter
from transformers import Sam3Processor, Sam3Model
from controlnet_aux import OpenposeDetector

REPO_ROOT = Path().absolute().parent

DEVICE = "cuda"
LAMBDA = 0.1
NUM_TRAIN = 4
LR = 5e-6
BATCH_SIZE = 1
UNIQUE_TOKEN = "sks"
OPENPOSE_WEIGHT = 0.8
SAM_PROMPT = "a humanoid robot"

INSTANCE_BASE_DIR = str(REPO_ROOT / "datasets" / "dataset")
PRIOR_BASE_DIR = str(REPO_ROOT / "datasets" / "dataset_prior")
MODEL_SAVE_DIR = str(REPO_ROOT / "outputs" / "saved_models" / "our_controlnet")

CLASSES_DICT = {
    'humanoid robot': ['unitree']
}

image_transforms = transforms.Compose([
    transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

cond_transforms = transforms.Compose([
    transforms.Resize((512, 512), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
])

mask_transforms = transforms.Compose([
    transforms.Resize((64, 64), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

class SimpleImageDataset(Dataset):
    def __init__(self, data_dir):
        os.makedirs(data_dir, exist_ok=True)
        self.image_paths = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        
    def __len__(self): 
        return max(1, len(self.image_paths))
        
    def __getitem__(self, idx): 
        if not self.image_paths:
            return Image.new("RGB", (512, 512), (0, 0, 0))
        return Image.open(self.image_paths[idx]).convert("RGB")

class InstancePoseMaskDataset(Dataset):
    def __init__(self, data_dir, det_op):
        self.image_paths = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")) and not f.endswith("_mask.png")]
        self.data = []
        for img_path in self.image_paths:
            img = Image.open(img_path).convert("RGB")
            mask_path = img_path.rsplit(".", 1)[0] + "_mask.png"
            mask_img = Image.open(mask_path).convert("L") if os.path.exists(mask_path) else Image.new("L", img.size, 255)
            self.data.append((
                image_transforms(img),
                cond_transforms(det_op(img)),
                mask_transforms(mask_img)
            ))
        
    def __len__(self): 
        return len(self.data)
        
    def __getitem__(self, idx): 
        return self.data[idx]

def class_collate_fn(examples):
    return torch.stack([image_transforms(img) if isinstance(img, Image.Image) else img for img in examples])

def instance_collate_fn(examples):
    return (torch.stack([e[0] for e in examples]), torch.stack([e[1] for e in examples]), torch.stack([e[2] for e in examples]))

def prepare_sam_masks(instance_dir):
    missing_masks = []
    for f in os.listdir(instance_dir):
        if f.lower().endswith((".jpg", ".jpeg", ".png")) and not f.endswith("_mask.png"):
            mask_name = f.rsplit(".", 1)[0] + "_mask.png"
            if not os.path.exists(os.path.join(instance_dir, mask_name)):
                missing_masks.append(f)

    if not missing_masks:
        return

    model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)
    processor = Sam3Processor.from_pretrained("facebook/sam3")

    for img_name in tqdm(missing_masks):
        img_path = os.path.join(instance_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        inputs = processor(images=image, text=SAM_PROMPT, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        results = processor.post_process_instance_segmentation(
            outputs, threshold=0.5, mask_threshold=0.5,
            target_sizes=inputs.get("original_sizes", torch.tensor([[image.height, image.width]])).tolist() 
        )[0]
        
        if len(results["masks"]) > 0:
            best_mask = results["masks"][0]
            if isinstance(best_mask, torch.Tensor): best_mask = best_mask.detach().cpu().numpy()
            mask_np = (np.squeeze(best_mask) > 0).astype(np.uint8) * 255
            Image.fromarray(mask_np).save(os.path.join(instance_dir, img_name.rsplit(".", 1)[0] + "_mask.png"))

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()

def train_and_save(class_name, instance):
    instance_dir = os.path.join(INSTANCE_BASE_DIR, instance)
    class_dir = os.path.join(PRIOR_BASE_DIR, instance) 
    save_path = os.path.join(MODEL_SAVE_DIR, instance)
    os.makedirs(save_path, exist_ok=True)
    
    prepare_sam_masks(instance_dir)
    
    instance_prompt = f"a {UNIQUE_TOKEN} {class_name}"
    class_prompt = f"a {class_name}"
    
    pipeline = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float32, safety_checker=None).to(DEVICE)
    vae, text_encoder, tokenizer, unet, noise_scheduler = pipeline.vae, pipeline.text_encoder, pipeline.tokenizer, pipeline.unet, pipeline.scheduler
    
    adapter = T2IAdapter.from_pretrained("TencentARC/t2iadapter_openpose_sd14v1", torch_dtype=torch.float32).to(DEVICE).eval()
    detector_openpose = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
    
    vae.requires_grad_(False).eval()
    adapter.requires_grad_(False).eval()
    text_encoder.requires_grad_(True).train()
    unet.requires_grad_(True).train()
    
    optimizer = torch.optim.AdamW(itertools.chain(unet.parameters(), text_encoder.parameters()), lr=LR, weight_decay=1e-2)
    
    inst_loader = DataLoader(InstancePoseMaskDataset(instance_dir, detector_openpose), batch_size=BATCH_SIZE, shuffle=True, collate_fn=instance_collate_fn)
    class_loader = DataLoader(SimpleImageDataset(class_dir), batch_size=BATCH_SIZE, shuffle=True, collate_fn=class_collate_fn)
    inst_iter, class_iter = cycle(inst_loader), cycle(class_loader)

    def tokenize(prompt): 
        return tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    
    inst_tokens, class_tokens = tokenize(instance_prompt), tokenize(class_prompt)

    for step in tqdm(range(1, NUM_TRAIN + 1)):
        optimizer.zero_grad()
        
        inst_pixels, inst_poses, inst_masks = next(inst_iter)
        class_pixels = next(class_iter)
        
        inst_poses, inst_masks = inst_poses.to(DEVICE), inst_masks.to(DEVICE)
        
        with torch.no_grad():
            latents = vae.encode(torch.cat([inst_pixels.to(DEVICE), class_pixels.to(DEVICE)])).latent_dist.sample() * vae.config.scaling_factor
            
            adapter_cond_pose = torch.cat([inst_poses, torch.zeros_like(inst_poses)], dim=0)
            adapter_states = adapter(adapter_cond_pose)
            if not isinstance(adapter_states, (list, tuple)): 
                adapter_states = adapter_states.to_tuple()
            adapter_states = [OPENPOSE_WEIGHT * state for state in adapter_states]

        prompt_embeds = torch.cat([text_encoder(inst_tokens)[0], text_encoder(class_tokens)[0]])
        
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=DEVICE).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        
        noise_pred = unet(
            noisy_latents, timesteps, encoder_hidden_states=prompt_embeds,
            down_intrablock_additional_residuals=[sample.to(dtype=torch.float32) for sample in adapter_states]
        ).sample
        
        noise_pred_inst, noise_pred_prior = noise_pred.chunk(2, dim=0)
        target_inst, target_prior = noise.chunk(2, dim=0)
        
        loss_inst_unreduced = F.mse_loss(noise_pred_inst, target_inst, reduction="none")
        loss_inst = (loss_inst_unreduced * inst_masks).mean()
        
        loss_prior = F.mse_loss(noise_pred_prior, target_prior, reduction="mean")
        loss = loss_inst + LAMBDA * loss_prior
        
        loss.backward()
        optimizer.step()

    pipeline.save_pretrained(save_path)
    
    del optimizer, inst_loader, class_loader, pipeline, adapter, detector_openpose
    torch.cuda.empty_cache()
    gc.collect()

if __name__ == "__main__":
    for class_name, instances in CLASSES_DICT.items():
        for instance in instances:
            train_and_save(class_name, instance)

/usr/local/lib/python3.12/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install 'mediapipe'
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/controlnet_aux/segment_anything/modeling/tiny_vit_sam.py:654: UserWarning: Overwriting tiny_vit_5m_224 in registry with controlnet_aux.segmen

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /workspace/.hf_home/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. F

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]